###Faisal Alsuqayri

---

Assignment 3

Use case: Customer Support Assistant

In [13]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage

In [1]:
import os
from google.colab import userdata

# We use OpenRouter for the agent — add OPENROUTER_API_KEY to Colab Secrets (key icon in left sidebar)
# Get your key at https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [2]:
!pip install "langchain[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


In [7]:
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [6]:
@tool
def get_order_status(order_id: str) -> str:
    """Check the shipping status of a specific order. Requires exact order ID."""
    return f"Order {order_id} is currently In Transit and expected to arrive tomorrow."

@tool
def process_refund(order_id: str, reason: str) -> str:
    """Process a refund for an order. Requires exact order ID and a reason."""
    return f"Refund initiated for order {order_id}. Reason logged: {reason}."


@tool
def search_knowledge_base(query: str) -> str:
    """Search the store's knowledge base for policies, sizing, and product info."""
    if "return" in query.lower():
        return "Return Policy: Items can be returned within 30 days of receipt in original packaging."
    elif "size" in query.lower():
        return "Sizing Guide: Our shoes run half a size small. We recommend sizing up."
    return "I couldn't find specific information on that in the knowledge base."

In [9]:
SUPERVISOR_PROMPT = (
    "You are a customer support coordinator for an e-commerce store. "
    "You can manage specific orders and answer general policy questions. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        manage_orders,
        answer_policy_questions
    ],
    system_prompt=SUPERVISOR_PROMPT,
)

In [10]:
ORDER_AGENT_PROMPT = (
    "You are an order management assistant. "
    "Parse natural language requests to extract order IDs and process them. "
    "Use get_order_status to check tracking and process_refund for returns. "
    "Always confirm the status or action taken in your final response."
)

order_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[get_order_status, process_refund],
    system_prompt=ORDER_AGENT_PROMPT,
)

POLICY_AGENT_PROMPT = (
    "You are a helpful customer policy assistant. "
    "Answer customer questions regarding store policies, sizing, or products. "
    "Use search_knowledge_base to find the correct information. "
    "Always provide a polite, clear answer based on the search results."
)

policy_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[search_knowledge_base],
    system_prompt=POLICY_AGENT_PROMPT,
)

In [11]:
@tool
def manage_orders(request: str) -> str:
    """Handle inquiries about specific orders, tracking status, and refunds.

    Use this when the user provides an order number or asks about the status
    of their purchase, shipping, or getting their money back.

    Input: Natural language request with order details (e.g., 'where is order #12345').
    """
    result = order_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

@tool
def answer_policy_questions(request: str) -> str:
    """Answer general questions about store policies, returns, sizing, or products.

    Use this when the user asks open-ended questions about how the store works,
    rules for returns, or general product information (no order ID required).

    Input: Natural language question (e.g., 'what is your return policy?').
    """
    result = policy_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

In [12]:
query = "Can you check the status of my order #98765? Also, if it doesn't fit, what is your return policy?"

result = supervisor_agent.invoke({"messages": [HumanMessage(query)]})

for message in result.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Can you check the status of my order #98765? Also, if it doesn't fit, what is your return policy?
================================== Ai Message ==================================
Tool Calls:
  manage_orders (call_69479a3920e543cd80b568d2)
 Call ID: call_69479a3920e543cd80b568d2
  Args:
    request: order #98765
================================= Tool Message =================================
Name: manage_orders

Your order **#98765** is currently **in transit** and is expected to arrive tomorrow. If you have any other questions or need further assistance, just let me know!
================================== Ai Message ==================================
Tool Calls:
  answer_policy_questions (call_731e020feaba44bda76c2a1f)
 Call ID: call_731e020feaba44bda76c2a1f
  Args:
    request: return policy
================================= Tool Message =================================
Name: answer_policy_questions

O